In [2]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
     

import torchvision
import torchvision.transforms as transforms

from models import *
use_gpu = torch.cuda.is_available()
batch_size = 128
model_name = "vgg16_quant_iter1"
model = VGG16_quant()

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])

In [3]:
train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))

trainloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))


testloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

In [9]:
print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)

        input = input.cuda(non_blocking=True)
        target = target.cuda(non_blocking=True)

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input = input.cuda(non_blocking=True)
            target = target.cuda(non_blocking=True)

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

        

In [4]:
def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [45, 90, 180]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.02      

In [5]:
import os
import glob
import shutil
import torch

def save_checkpoint(state, is_best, fdir, best_prec):
    os.makedirs(fdir, exist_ok=True)
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)

    if is_best:
        # Remove previous best checkpoint
        for old_file in glob.glob(os.path.join(fdir, 'model_best_prec=*.pth.tar')):
            os.remove(old_file)

        # Save new best checkpoint with accuracy in filename
        best_path = os.path.join(fdir, f'model_best_prec={best_prec:.3f}.pth.tar')
        shutil.copyfile(filepath, best_path)


In [6]:
model_name = "vgg16_qat"
learning_rate = 1e-4
weight_decay = 2e-2
model_name = model_name + f"_lr={learning_rate}_wd={weight_decay}"
print(model_name)

vgg16_qat_lr=0.0001_wd=0.02


In [7]:
model_name = "vgg16_qat"
cudnn.benchmark = True
criterion = nn.CrossEntropyLoss().cuda()

def MFHPO_LR_WD(learning_rate, weight_decay, max_epochs):
    model = VGG16_quant()
    best_prec = 0
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=weight_decay)
    model.cuda()
    model_name_new = model_name + f"_lr={learning_rate}_wd={weight_decay}"
    fdir = 'result_checkpoints/'+str(model_name_new)
    if not os.path.exists(fdir):
        os.makedirs(fdir)
    for epoch in range(0, max_epochs):
        adjust_learning_rate(optimizer, epoch)
    
        train(trainloader, model, criterion, optimizer, epoch)
        
        # evaluate on test set
        print("Validation starts")
        prec = validate(testloader, model, criterion)
    
        # remember best precision and save checkpoint
        is_best = prec > best_prec
        best_prec = max(prec,best_prec)
        print('best acc: {:1f}'.format(best_prec))
        save_checkpoint({
            'epoch': epoch + 1,
            'state_dict': model.state_dict(),
            'best_prec': best_prec,
            'optimizer': optimizer.state_dict(),
        }, is_best, fdir, best_prec)
    return best_prec

In [17]:
lr = 0.05
wd = 0.02
lw_dict = {}
lw_dict[(lr,wd)] = 0.1
print(lw_dict)

{(0.05, 0.02): 0.1}


In [ ]:
import pandas as pd
model_name = "vgg16_qat"
cudnn.benchmark = True
criterion = nn.CrossEntropyLoss().cuda()
learning_rates = [
    0.0025, 0.005, 0.01, 0.02,
    0.04, 0.06, 0.10, 0.16
]

weight_decays = [
    0.0, 1e-5, 1e-4, 5e-4, 2e-3
]
max_epochs = 5
best_prec = 0

results = []

for lr in learning_rates[:4]:
    for wd in weight_decays:
        best_prec = MFHPO_LR_WD(lr, wd, max_epochs)

        results.append({
            'lr': lr,
            'weight_decay': wd,
            'best_prec': best_prec
        })

        # Save progress after every trial
        df = pd.DataFrame(results)
        df.to_csv('sweep_1_results.csv', index=False)

# Sort by highest accuracy
df = df.sort_values(by='best_prec', ascending=False)
df = df.reset_index(drop=True)


display(df)
            

Epoch: [0][0/391]	Time 24.305 (24.305)	Data 15.745 (15.745)	Loss 2.5075 (2.5075)	Prec 8.594% (8.594%)
Epoch: [0][100/391]	Time 0.021 (0.268)	Data 0.000 (0.156)	Loss 1.6378 (1.8417)	Prec 39.062% (31.149%)
Epoch: [0][200/391]	Time 0.023 (0.148)	Data 0.000 (0.079)	Loss 1.3570 (1.7196)	Prec 46.875% (36.120%)
Epoch: [0][300/391]	Time 0.022 (0.108)	Data 0.000 (0.053)	Loss 1.3109 (1.6291)	Prec 49.219% (39.714%)
Validation starts
Test: [0/79]	Time 12.507 (12.507)	Loss 1.2297 (1.2297)	Prec 56.250% (56.250%)


In [1]:
import pandas as pd
print(pd.__version__)

2.3.3
